**About** : This notebook is used to infer models.

In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
cd ../src/

/workspace/kaggle_rsna_abdominal_trauma/src


### Imports

In [ ]:
# !pip install -qU python-gdcm pydicom pylibjpeg dicomsdl

In [4]:
import os
import gc
import re
import sys
import cv2
import glob
import json
import torch
import warnings
import numpy as np
import pandas as pd
import torch.nn.functional as F
import matplotlib.pyplot as plt

from tqdm import tqdm

In [5]:
from inference.extract_features import Config
from inference.lvl2 import predict as predict_2
from inference.lvl2 import PatientFeatureInfDataset, to_sub_format
from inference.crop import get_crops

from util.torch import load_model_weights
from util.plots import plot_mask

from data.transforms import get_transfos
from data.dataset import AbdominalCropDataset

from inference.processing import process, restrict_imgs
from inference.lvl1 import predict, AbdominalInfDataset


from model_zoo.models import define_model
from model_zoo.models_lvl2 import define_model as define_model_2
from model_zoo.models_seg import define_model as define_model_seg
from model_zoo.models_seg import convert_3d

In [6]:
# dfp, dfi = prepare_data()

# dfg = dfi[dfi.bowel_healthy == False][["patient_id", "bowel_healthy", "bowel_injury"]].groupby('patient_id').max()
# dfg[dfg.bowel_injury == 0]

# dfg = dfi[dfi.extravasation_healthy == False][["patient_id", "extravasation_healthy", "extravasation_injury"]].groupby('patient_id').max()
# dfg[dfg.extravasation_injury == 0]

### Params

In [7]:
EVAL = True

In [8]:
DATA_PATH = "../input/test_images/"
SAVE_FOLDER = "../output/tmp/"

In [9]:
BATCH_SIZE = 32
BATCH_SIZE_2 = 512
USE_FP16 = True
NUM_WORKERS = 2

RESTRICT = False
HALF = True

In [10]:
FOLD = 0
# FOLD = "fullfit_0"

EXP_FOLDERS = [
    ("../output/2023-09-20_14/", "seg", [FOLD]),
#     ("../logs/2024-02-19/4/", "probas_2d", [FOLD]),  # 0.346  - maxvit_tiny_tf_512   NO SMH
#     ("../output/2023-10-10_27/", "probas_2d", [FOLD]),  # 0.346  - maxvit_tiny_tf_512
    ("../logs/2024-05-21/5/", "probas_2d", [FOLD]),  # 0.346  - maxvit_tiny_tf_512   NO SMH FIX
]

CROP_EXP_FOLDERS = [
#     ("../logs/2024-02-19/3/", "crop", [FOLD]), # coatnet_1_rw_224   -> 0.320 NO SMH
#     ("../output/2023-10-05_31/", "crop", [FOLD]),  # coatnet_1_rw_224   -> 0.320
#     ("../logs/2024-03-21/4/", "crop", [FOLD]), # coatnet_1_rw_224 NO SMH REPRO
     ("../logs/2024-05-21/7/", "crop", [FOLD]), # coatnet_1_rw_224 NO SMH REPRO FIX
]

EXP_FOLDERS_2 = [
#     "../logs/2024-03-21/5/",
#     "../logs/2024-03-21/6/",
#     "../logs/2024-05-22/0/",     # 0.324  -  FIX LABELS - Spleen
#     "../logs/2024-09-20/1/",  # 0.123 / 0.469 - Bowel extrav
    "../logs/2024-10-03/0/",
]
EXP_FOLDER_3D = "../output/2023-09-24_20/"

FOLDS_2 = [0, 1, 2, 3]
if "fullfit" not in str(FOLD):
    FOLDS_2 = [FOLD]

    
for f in EXP_FOLDERS_2:
    folders = Config(json.load(open(f + "config.json", 'r'))).exp_folders
    print(f)
    print([f[0] for f in folders if "probas" in f[1]])
    print([f[0] for f in folders if "crop" in f[1]])
    print()
    
if EVAL:
    df = pd.read_csv("../logs/2024-02-19/4/df_val_0.csv")
    DATA_PATH = f"../input/train_images/"
    patients = [318, 156, 310, 318, 403]  # sorted(df['patient_id'].unique())[:10]
#     patients = sorted(df['patient_id'].unique())
#     print(patients)

../logs/2024-10-03/0/
['../logs/2024-05-21/5/']
['../logs/2024-05-21/7/']



In [11]:
# import shutil
# for pid in [
#     10082, 10004, 10005, 10007,
# ]:
#     try:
#         shutil.copytree(f"../input/train_images/{pid}", DATA_PATH + f"{pid}/")
#     except FileExistsError:
#         pass

In [16]:
from util.logger import upload_to_kaggle

upload_to_kaggle(
    [
#         "../logs/2024-05-22/0/",  # 0.324  -  FIX LABELS
#         "../logs/2024-05-21/7/",  # coatnet_1_rw_224 -1 11 fix labels
#         "../logs/2024-05-21/5/",  # maxvit_tiny_tf_512 fix labels
        "../logs/2024-10-03/0/",  # 0.120 / 0.470 - Bowel extrav  No restrict no maxlen
    ],
    "../output/weights_spleen_fix/",
    "RSNA Weights spleen fix"
)

- Copying ../logs/2024-10-03/0/ ...

Dataset size : 2.560 Go
- Update existing dataset !
- Uploading ...


100%|██████████| 38.1M/38.1M [00:02<00:00, 18.2MB/s]
100%|██████████| 752M/752M [00:30<00:00, 25.7MB/s]    
100%|██████████| 237M/237M [00:02<00:00, 91.6MB/s]   
100%|██████████| 30.6M/30.6M [00:02<00:00, 15.3MB/s]
100%|██████████| 567M/567M [00:05<00:00, 111MB/s]    
100%|██████████| 756M/756M [00:35<00:00, 22.5MB/s]   
100%|██████████| 38.1M/38.1M [00:02<00:00, 17.9MB/s]



Output :
 b'Starting upload for file 2024-10-03_0.zip\nUpload successful: 2024-10-03_0.zip (38MB)\nStarting upload for file 2024-05-21_7.zip\nUpload successful: 2024-05-21_7.zip (752MB)\nStarting upload for file 2023-09-20_14.zip\nUpload successful: 2023-09-20_14.zip (237MB)\nStarting upload for file 2024-05-22_0.zip\nUpload successful: 2024-05-22_0.zip (31MB)\nStarting upload for file 2024-05-21_5.zip\nUpload successful: 2024-05-21_5.zip (567MB)\nStarting upload for file 2023-09-24_20.zip\nUpload successful: 2023-09-24_20.zip (756MB)\nStarting upload for file 2024-09-20_1.zip\nUpload successful: 2024-09-20_1.zip (38MB)\nDataset version is being created. Please check progress at https://www.kaggle.com/theoviel/rsna-weights-spleen-fix\n'

Error :
 None


## Seg & Level 1

In [13]:
class Config3d:
    size = 256
    plot = FOLD == 0
    margin = 5
#     n_frames_to_save = 30

config = Config(json.load(open(EXP_FOLDER_3D + "config.json", "r")))

model_seg = define_model_seg(
    config.decoder_name,
    config.name,
    num_classes=config.num_classes,
    num_classes_aux=config.num_classes_aux,
    n_channels=config.n_channels,
    increase_stride=config.increase_stride,
    pretrained=False,
)

model_seg = convert_3d(model_seg)
model_seg = load_model_weights(model_seg, EXP_FOLDER_3D + f"{config.name}_{FOLD}.pt")
model_seg = model_seg.eval().cuda()


 -> Loading encoder weights from ../output/2023-09-24_20/resnet18d_0.pt



In [14]:
models = []

for exp_folder, mode, folds in EXP_FOLDERS:
    models_ = []
    config = Config(json.load(open(exp_folder + "config.json", "r")))

    for fold in folds:
        model = define_model(
            config.name,
            drop_rate=config.drop_rate,
            drop_path_rate=config.drop_path_rate,
            use_gem=config.use_gem,
            head_3d=config.head_3d if hasattr(config, "head_3d") else "",
            n_frames=config.n_frames if hasattr(config, "n_frames") else "",
            num_classes=config.num_classes,
            num_classes_aux=config.num_classes_aux,
            n_channels=config.n_channels,
            reduce_stride=config.reduce_stride,
            increase_stride=config.increase_stride if hasattr(config, "increase_stride") else False,
            pretrained=False
        )
        model = model.cuda().eval()
    
        weights = exp_folder + f"{config.name}_{fold}.pt"
        model = load_model_weights(model, weights, verbose=config.local_rank == 0)
        models_.append(model)
        
    models.append(models_)


 -> Loading encoder weights from ../output/2023-09-20_14/efficientnetv2_rw_t_0.pt


 -> Loading encoder weights from ../logs/2024-05-21/5/maxvit_tiny_tf_512_0.pt



In [15]:
# %%time

dfs = []
for patient in tqdm(sorted(os.listdir(DATA_PATH))):
    if FOLD == 0:
        if EVAL:
            if int(patient) not in patients:
                continue
        else:
            if patient != "10082":
                continue
    elif FOLD == 3:
        if patient != "10004":
            continue

    for series in sorted(os.listdir(DATA_PATH + patient)):
        print("-> Patient", patient, '- Series', series)

        imgs, paths, n_imgs = process(
            patient,
            series,
            data_path=DATA_PATH,
            on_gpu=True,
            crop_size=384,
            restrict=RESTRICT
        )

        # Seg & Crop
        with torch.cuda.amp.autocast(enabled=True):
            x = F.interpolate(
                imgs.unsqueeze(0).unsqueeze(0),
                size=(Config3d.size, Config3d.size, Config3d.size),
                mode="nearest"
            )
            pred = model_seg(x)[0].argmax(1, keepdims=True).float()
            pred = F.interpolate(pred, size=(len(imgs), 384, 384), mode="nearest")

        seg = pred[0][0]
#         break
        coords = get_crops(seg)

        for (x0, x1, y0, y1, z0, z1), name in zip(coords, ['liver', 'spleen', 'kidney']):
            x0, x1 = max(0, x0 - Config3d.margin), min(imgs.shape[0], x1 + Config3d.margin)
            y0, y1 = max(0, y0 - Config3d.margin), min(imgs.shape[1], y1 + Config3d.margin)
            z0, z1 = max(0, z0 - Config3d.margin), min(imgs.shape[2], z1 + Config3d.margin)

            img_crop = (imgs[x0: x1, y0:y1, z0:z1].cpu().numpy() * 255).astype(np.uint8)
            np.save(SAVE_FOLDER + f'{patient}_{series}_{name}.npy', img_crop.copy())

        # Cls
        df = pd.DataFrame({"path": paths})
        df['patient_id'] = df['path'].apply(lambda x: x.split('_')[0])
        df['patient'] = df['path'].apply(lambda x: x.split('_')[0])
        df['series'] = df['path'].apply(lambda x: x.split('_')[1])
        df['frame'] = df['path'].apply(lambda x: int(x.split('_')[2][:-4]))
        dfs.append(df)

        for models_list, (exp_folder, t, _) in zip(models, EXP_FOLDERS):
            exp_name = "_".join(exp_folder.split('/')[-2:-1])
            if "2023" not in exp_name:  # locally
                exp_name = "_".join(exp_folder.split('/')[-3:-1])
            
            config = Config(json.load(open(exp_folder + "config.json", "r")))

            dataset = AbdominalInfDataset(
                df,
                frames_chanel=config.frames_chanel if hasattr(config, "frames_chanel") else 0,
                n_frames=config.n_frames if hasattr(config, "n_frames") else 1,
                stride=config.stride if hasattr(config, "stride") else 1,
                imgs=imgs,
                paths=paths,
            )
            if HALF:
                dataset.info = dataset.info[::2]

            preds = []
            for model in models_list:
                pred = predict(
                    model,
                    dataset,
                    config.loss_config,
                    batch_size=BATCH_SIZE,
                    use_fp16=USE_FP16,
                    num_workers=0,
                    resize=config.resize if config.resize[0] != 384 else None
                )
#                     print(pred.shape)
                if HALF:
                    pred = np.repeat(pred, 2, axis=0)[:len(df)]
#                     print(pred.shape)
                preds.append(pred)

            if RESTRICT:
                pred_padded = np.zeros((n_imgs, pred.shape[-1]))
                pred_padded[-len(pred):] = np.mean(preds, 0)
            else:
                pred_padded = np.mean(preds, 0)

            np.save(SAVE_FOLDER + f"{series}_{exp_name}.npy", pred_padded)
            
            if FOLD == 0 and not RESTRICT:
                ref = np.load(exp_folder + "pred_val_0.npy")
                ref = ref[:len(pred)]
                
                plt.plot(ref - np.mean(preds, 0))
                plt.title(f'{t} - {np.abs(ref - np.mean(preds, 0)).max()}')
                plt.show()

#             break
#     break

df = pd.concat(dfs, ignore_index=True)
df = df.groupby(['patient', 'series']).max().reset_index()

# del model_seg, models, imgs, x, pred, seg, dataset
torch.cuda.empty_cache()
gc.collect()

FileNotFoundError: [Errno 2] No such file or directory: '../input/train_images/'

In [ ]:
# patient_id, series = 318, 58344

# for o in ['kidney', 'liver', 'spleen']:
#     ref_spleen = np.load("../input/crops/imgs_2/" + f"{patient_id}_{series}_{o}.npy")
#     spleen = np.load("../output/tmp/" + f"{patient_id}_{series}_{o}.npy")
#     print(o, ref_spleen.shape, spleen.shape)

# ref_seg = np.load("../input/masks/" + f"mask_{patient_id}_{series}.npy")

# coords_ref = get_crops(torch.from_numpy(ref_seg))
# for (x0, x1, y0, y1, z0, z1), name in zip(coords_ref, ['liver', 'spleen', 'kidney']):
#     x0, x1 = max(0, x0 - Config3d.margin), min(imgs.shape[0], x1 + Config3d.margin)
#     y0, y1 = max(0, y0 - Config3d.margin), min(imgs.shape[1], y1 + Config3d.margin)
#     z0, z1 = max(0, z0 - Config3d.margin), min(imgs.shape[2], z1 + Config3d.margin)
    
#     print(name, end="\t")
#     print(
#         - (x0 - x1),
#         - (y0 - y1),
#         - (z0 - z1)
#     )
# print()
    
# for (x0, x1, y0, y1, z0, z1), name in zip(get_crops(seg), ['liver', 'spleen', 'kidney']):
#     x0, x1 = max(0, x0 - Config3d.margin), min(imgs.shape[0], x1 + Config3d.margin)
#     y0, y1 = max(0, y0 - Config3d.margin), min(imgs.shape[1], y1 + Config3d.margin)
#     z0, z1 = max(0, z0 - Config3d.margin), min(imgs.shape[2], z1 + Config3d.margin)

#     print(name, end="\t")
# #     print(x0, x1)
# #     print(y0, y1)
# #     print(z0, z1)
#     print(
#         - (x0 - x1),
#         - (y0 - y1),
#         - (z0 - z1)
#     )

In [ ]:
# for i in range(0, len(spleen), 10):
#     plt.subplot(1, 2, 1)
#     plt.imshow(spleen[i], cmap="gray")
#     plt.axis(False)
#     plt.subplot(1, 2, 2)
#     plt.imshow(ref_spleen[i], cmap="gray")
#     plt.axis(False)
#     plt.show()
#     break

## Crop models

In [ ]:
df_series = pd.DataFrame({"img_path": sorted(glob.glob(SAVE_FOLDER + f'*.npy'))})
df_series['patient_id'] = df_series['img_path'].apply(lambda x: x.split('/')[-1].split('_')[0])
df_series['series'] = df_series['img_path'].apply(lambda x: x.split('_')[-2])
df_series['organ'] = df_series['img_path'].apply(lambda x: x.split('_')[-1][:-4])

df_series['target'] = 0
df_series = df_series[df_series['organ'].isin(['kidney', 'liver', 'spleen'])].reset_index(drop=True)

if FOLD == 0 and not EVAL:
    df_series = df_series[df_series['patient_id'].astype(int) == 10082].reset_index(drop=True)
elif FOLD == 3:
    df_series = df_series[df_series['patient_id'].astype(int) == 10004].reset_index(drop=True)
    
# CHECK REPRO
# df_series['img_path'] = df_series['img_path'].apply(lambda x: re.sub('../output/tmp/', '../input/crops/imgs_2/', x))

In [ ]:
crop_fts = []
for exp_folder, mode, folds in CROP_EXP_FOLDERS:
    
    config = Config(json.load(open(exp_folder + "config.json", "r")))

    model = define_model(
        config.name,
        drop_rate=config.drop_rate,
        drop_path_rate=config.drop_path_rate,
        use_gem=config.use_gem,
        head_3d=config.head_3d if hasattr(config, "head_3d") else "",
        n_frames=config.n_frames if hasattr(config, "n_frames") else "",
#         replace_pad_conv=config.replace_pad_conv,
        num_classes=config.num_classes,
        num_classes_aux=config.num_classes_aux,
        n_channels=config.n_channels,
        reduce_stride=config.reduce_stride,
        increase_stride=config.increase_stride if hasattr(config, "increase_stride") else False,
        pretrained=False
    )
    model = model.cuda().eval()
    
    preds = []
    for fold in folds:
        weights = exp_folder + f"{config.name}_{fold}.pt"
        model = load_model_weights(model, weights, verbose=config.local_rank == 0)
        
        transfos = get_transfos(
            augment=False, resize=config.resize, crop=config.crop
        )

        dataset = AbdominalCropDataset(
            None,
            None,
            transforms=transfos,
            frames_chanel=config.frames_chanel,
            n_frames=config.n_frames,
            stride=config.stride,
            train=False,
            df_series=df_series
        )

        pred = predict(
            model,
            dataset,
            config.loss_config,
            batch_size=BATCH_SIZE,
            use_fp16=USE_FP16,
            num_workers=NUM_WORKERS,
        )
        preds.append(pred)

    preds = np.mean(preds, 0)
    crop_fts.append(preds)
    
crop_fts = np.array(crop_fts) # n_models x 3*n_studies x n_classes
np.save(SAVE_FOLDER + "crop_fts.npy", crop_fts)

In [ ]:
# display(df_series.head(10))
# crop_fts

In [ ]:
crop_fts = np.load(SAVE_FOLDER + "crop_fts.npy")
crop_fts = crop_fts.reshape(crop_fts.shape[0], crop_fts.shape[1] // 3, 3, crop_fts.shape[2])  # n_models x n_studies x n_organs x 3
crop_fts = crop_fts.transpose(1, 2, 0, 3)  # n_studies x n_organs x n_models x 3
crop_fts = crop_fts.reshape(crop_fts.shape[0], crop_fts.shape[1], crop_fts.shape[2] * crop_fts.shape[3])  # n_studies x n_organs x 3 * n_models

## Level 2

In [ ]:
all_preds = []

for exp_folder in EXP_FOLDERS_2:
    config_2 = Config(json.load(open(exp_folder + "config.json", "r")))
    config_2.exp_folders = [[re.sub('_', '/', f[0]), f[1]] for f in config_2.exp_folders]

    dataset = PatientFeatureInfDataset(
        df['series'],
        config_2.exp_folders,
        crop_fts=crop_fts,
        max_len=config_2.max_len,
        restrict=config_2.restrict,
        resize=config_2.resize,
        save_folder=SAVE_FOLDER,
        half=HALF,
    )
    
    model = define_model_2(
        config_2.name,
        ft_dim=config_2.ft_dim,
        layer_dim=config_2.layer_dim,
        n_layers=config_2.n_layers,
        dense_dim=config_2.dense_dim,
        p=config_2.p,
        num_classes=config_2.num_classes,
        num_classes_aux=config_2.num_classes_aux,
        n_fts=config_2.n_fts,
    )
    model = model.eval().cuda()

    for fold in FOLDS_2:
        weights = exp_folder + f"{config_2.name}_{fold}.pt"
        model = load_model_weights(model, weights, verbose=1, strict=True)

        preds = predict_2(
            model,
            dataset,
            config_2.loss_config,
            batch_size=BATCH_SIZE_2,
            use_fp16=USE_FP16,
            num_workers=NUM_WORKERS,
        )
        all_preds.append(preds)

In [ ]:
preds = np.mean(all_preds, 0).astype(np.float64)

for i in range(preds.shape[1]):
    df[f'pred_{i}'] = preds[:, i]

dfg = df.drop(['series', 'path', 'frame', 'patient_id'], axis=1).groupby('patient').mean().reset_index()
sub = to_sub_format(dfg)

sub.head()

In [ ]:
# sub.to_csv(SAVE_FOLDER + "submission_.csv", index=False)
# sub

### Checks

In [ ]:
from training.main_lvl2 import k_fold, retrieve_preds
from inference.extract_features import Config as ConfigInf
from data.preparation import prepare_data, get_df_series
from util.metrics import rsna_loss

EXP_FOLDERS = [  # NO SMH - Spleen scores
    "../logs/2024-03-21/5/",   # 0.350    - spleen_only=True spleen model reprocrop    <--
]

In [ ]:
df_patient, df_img = prepare_data("../input/")

In [ ]:
if "fold" not in df_patient.columns:
    folds = pd.read_csv(config.folds_file)
    df_img = df_img.merge(folds)
    df_patient = df_patient.merge(folds)

In [ ]:
preds_oof = []
for exp_folder in EXP_FOLDERS:
    cfg = ConfigInf(json.load(open(exp_folder + "config.json", "r")))
    print(
        exp_folder, " --> ",
        "proba :",
        " - ".join(e[8:] for e, m in cfg.exp_folders if "proba" in m),
        "\t crop : ",
        " - ".join(e[8:] for e, m in cfg.exp_folders if "crop" in m),
        "\n"
    )

    df_oof, pred_oof = retrieve_preds(
        df_patient,
        df_img,
        cfg,
        exp_folder,
        custom_agg=False,
        folds=[0]
    )
    preds_oof.append(pred_oof)
    
    losses, avg_loss = rsna_loss(pred_oof, df_oof)

    for k, v in losses.items():
        if k == "spleen":
            print(f"- {k.split('_')[0][:8]} loss\t: {v:.3f}")

#     print(f'\n -> CV Score : {avg_loss :.4f}')

#     df_oof.to_csv(exp_folder + 'df_oof.csv', index=False)

pred_oof = np.mean(preds_oof, 0)

In [ ]:
sub.patient_id = sub.patient_id.astype(int)
df_oof = df_oof.merge(sub, on="patient_id", suffixes=("", "_pred"))

losses, avg_loss = rsna_loss(df_oof[df_oof.columns[-11:]].values, df_oof)

for k, v in losses.items():
    if k == "spleen":
        print(f"- {k.split('_')[0][:8]} loss\t: {v:.3f}")

In [ ]:
sub.patient_id = sub.patient_id.astype(int)
df_oof = df_oof.merge(sub, on="patient_id", suffixes=("", "_pred"))

losses, avg_loss = rsna_loss(df_oof[df_oof.columns[-11:]].values, df_oof)

for k, v in losses.items():
    if k == "spleen":
        print(f"- {k.split('_')[0][:8]} loss\t: {v:.3f}")

In [ ]:
sub.patient_id = sub.patient_id.astype(int)
df_oof = df_oof.merge(sub, on="patient_id", suffixes=("", "_pred"))

losses, avg_loss = rsna_loss(df_oof[df_oof.columns[-11:]].values, df_oof)

for k, v in losses.items():
    if k == "spleen":
        print(f"- {k.split('_')[0][:8]} loss\t: {v:.3f}")

In [ ]:
df_oof = pd.read_csv(exp_folder + 'df_oof.csv')
df_oof = df_oof[df_oof['patient_id'].isin(sub['patient_id'].values.astype(int))]

df_oof = df_oof[["patient_id", "fold"] + list(df_oof.columns[-11:])]
df_oof.columns = ["patient_id", "fold", "bowel_injury", "extravasation_injury"] + list(sub.columns[-9:])
df_oof["extravasation_healthy"] = 1 - df_oof["extravasation_injury"]
df_oof["bowel_healthy"] = 1 - df_oof["bowel_injury"]

df_oof = df_oof.sort_values('patient_id', ignore_index=True)

df_oof['diff'] = (sub[sub.columns[1:]] - df_oof[sub.columns[1:]]).abs().max(1)

df_oof = df_oof[list(sub.columns) + ['fold', 'diff']]
# df_oof[df_oof['fold'] == FOLD]
df_oof

Done ! 